In [2]:
# imports
import snntorch as snn
from snntorch import surrogate
from snntorch import backprop
from snntorch import functional as SF
from snntorch import utils
from snntorch import spikeplot as splt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
import itertools

ModuleNotFoundError: No module named 'torch'

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=1e-2, betas=(0.9, 0.999))
num_epochs = 5
loss_hist = []
test_acc_hist = []
counter = 0

for epoch in range(num_epochs):

    net.train()
    for data, targets in iter(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        # Forward pass through time
        spk_rec, _ = forward_pass(net, num_steps, data)

        # Compute mean loss across time steps
        loss_val = loss_fn(spk_rec, targets).mean()

        # Backprop + update
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()

        loss_hist.append(loss_val.item())

        if counter % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{counter}], Loss: {loss_val.item():.4f}")

        if counter % 200 == 0:
            with torch.no_grad():
                net.eval()
                test_acc = batch_accuracy(test_loader, net, num_steps)
                print(f"→ Iter {counter}: Test Acc = {test_acc * 100:.2f}%")
                test_acc_hist.append(test_acc.item())

        counter += 1
